# Run Research Agents with Gemma 4 on Kaggle

Before running: enable a GPU accelerator, turn on Internet in the Kaggle notebook settings, and add your Hugging Face token as a Kaggle secret named `HF_TOKEN`. If your pipeline uses E2B, also add `E2B_API_KEY`.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "transformers": "transformers>=4.52.0",
    "torch": "torch>=2.4.0",
    "accelerate": "accelerate>=0.34.0",
    "bitsandbytes": "bitsandbytes>=0.43.0",
    "sentence_transformers": "sentence-transformers>=3.0.0",
    "chromadb": "chromadb>=0.5.0",
    "arxiv": "arxiv>=2.1.0",
    "pydantic": "pydantic>=2.7.0",
    "pydantic_settings": "pydantic-settings>=2.3.0",
    "loguru": "loguru>=0.7.2",
    "fpdf": "fpdf2>=2.7.9",
    "markdown2": "markdown2>=2.4.13",
    "dotenv": "python-dotenv>=1.0.0",
    "google.genai": "google-genai>=1.0.0",
    "e2b_code_interpreter": "e2b-code-interpreter>=0.0.10",
}

missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing or "none")

if missing:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "pip could not download packages. In Kaggle, open Notebook options and enable Internet. "
            "If Internet is already enabled, restart the session and run this cell again."
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import os
import shutil
import sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    try:
        os.environ["E2B_API_KEY"] = secrets.get_secret("E2B_API_KEY")
    except Exception:
        os.environ.setdefault("E2B_API_KEY", "")
except Exception as exc:
    print(f"Kaggle secrets not available yet: {exc}")

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path("/kaggle/working")]
    candidates.extend(Path("/kaggle/input").glob("**/research_agents"))
    for candidate in candidates:
        if (candidate / "agents").exists() and (candidate / "config.py").exists():
            return candidate
        nested = candidate / "research_agents"
        if (nested / "agents").exists() and (nested / "config.py").exists():
            return nested
    raise FileNotFoundError("Could not find the research_agents project under /kaggle/input or /kaggle/working.")

source_root = find_project_root()
work_root = Path("/kaggle/working/research_agents")
if source_root.is_relative_to(Path("/kaggle/input")):
    if work_root.exists():
        shutil.rmtree(work_root)
    shutil.copytree(source_root, work_root, ignore=shutil.ignore_patterns(".git", "__pycache__", ".pytest_cache", "outputs"))
    project_root = work_root
else:
    project_root = source_root

os.chdir(project_root)
sys.path.insert(0, str(project_root.parent))
print(f"Running from writable project copy: {project_root}")

env_text = f"""
LLM_PROVIDER=huggingface
HF_TOKEN={os.environ.get('HF_TOKEN', '')}
E2B_API_KEY={os.environ.get('E2B_API_KEY', '')}
AGENT_MODEL=google/gemma-4-31B-it
REVIEWER_MODEL=google/gemma-4-31B-it
HF_LOAD_IN_4BIT=true
HF_TORCH_DTYPE=auto
HF_ATTN_IMPLEMENTATION=auto
MAX_OUTPUT_TOKENS=1024
MAX_ARXIV_RESULTS=5
MAX_ARXIV_QUERIES=1
ARXIV_DELAY_SECONDS=12
MAX_REVISION_CYCLES=1
CHROMA_PERSIST_DIR=/kaggle/working/storage/chroma
OUTPUTS_DIR=/kaggle/working/outputs
LOG_LEVEL=INFO
""".strip()

(project_root / ".env").write_text(env_text)
print(f"Wrote .env for Gemma 4 Kaggle run at {project_root / '.env'}")

In [ ]:
# Quick Gemma 4 smoke test. This loads the 31B model once; the research agents reuse the cached model in-process.
from research_agents.agents.plan_formulation.agent import PlanFormulationAgent

agent = PlanFormulationAgent()
print(agent.call_llm([{"role": "user", "content": "Reply with exactly: Gemma 4 ready"}], response_model=None))

In [ ]:
from research_agents.orchestrator import ResearchOrchestrator
from research_agents.utils.logger import setup_logger

setup_logger()
question = "How can multi-agent LLM systems improve scientific literature review quality?"
orchestrator = ResearchOrchestrator(research_question=question)
state = orchestrator.run()

print("Research complete")
print("Paper:", state.paper.pdf_path if state.paper else "not generated")
print("Outputs:", "/kaggle/working/outputs")